In [ ]:
# Courtesy of CHatGPT

In [29]:
# RDKit example:
# 1) create ibuprofen
# 2) BRICS decompose
# 3) save parent + each fragment as separate SVG files

import os
from rdkit import Chem
from rdkit.Chem import BRICS, Draw
from pathlib import Path
from IPython.display import SVG

# ---------- helpers ----------
def mol_to_svg_file(mol: Chem.Mol, out_path: Path, legend: str = "", size=(-1, -1)) -> None:
    """Write a single molecule to an SVG file."""
    if mol is None:
        raise ValueError(f"mol is None for output {out_path}")
    mol2 = Chem.Mol(mol)  # copy

    d2d = Draw.MolDraw2DSVG(size[0], size[1])
    dopts = d2d.drawOptions()
    dopts.useBWAtomPalette()
    dopts.clearBackground = False
    # we need to set the highlights to be circles or we'll end up with ovals
    # that fit around the atomic symbol

    # we need to provide highlightBonds=[] here to avoid having the bonds between highlighted atoms highlighted:
    d2d.DrawMolecule(mol2)
    d2d.FinishDrawing()
    with open(out_path, "w") as f:
        f.write(SVG(d2d.GetDrawingText()).data)

    

# ---------- 1) ibuprofen ----------
# Common SMILES for ibuprofen:
ibu_smiles = "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O"
ibu = Chem.MolFromSmiles(ibu_smiles)
if ibu is None:
    raise RuntimeError("Failed to build ibuprofen from SMILES")

# Output folder
out_dir = Path("ibuprofen_brics_svgs")
out_dir.mkdir(parents=True, exist_ok=True)

# Save whole molecule
mol_to_svg_file(ibu, out_dir / "ibuprofen.svg", legend="Ibuprofen")

# ---------- 2) BRICS decomposition ----------
# BRICSDecompose returns a *set of SMILES* for the fragments (with BRICS dummy atoms like [*])
frag_smiles_set = BRICS.BRICSDecompose(ibu)

# Convert fragment SMILES -> Mol objects
frags = []
for s in sorted(frag_smiles_set):
    m = Chem.MolFromSmiles(s)
    if m is None:
        # Some RDKit builds can be picky about dummy labels; sanitize=False fallback helps
        m = Chem.MolFromSmiles(s, sanitize=False)
        if m is not None:
            Chem.SanitizeMol(m)
    if m is not None:
        frags.append((s, m))


# clean frags

for sm, mol in frags:
    for at in mol.GetAtoms():
        if at.GetSymbol() == "*":
            at.SetIsotope(0) 

# ---------- 3) save each fragment as SVG ----------
for i, (smi, mol) in enumerate(frags, start=1):
    # Make filenames safe-ish
    safe = smi.replace("/", "_").replace("\\", "_").replace(":", "_").replace("*", "star")
    safe = "".join(ch if ch.isalnum() or ch in "._-" else "_" for ch in safe)
    mol_to_svg_file(
        mol,
        out_dir / f"fragment_{i:02d}.svg",
        legend=f"BRICS frag {i}",
        size=(-1, -1),
    )

print(f"Wrote: {out_dir/'ibuprofen.svg'}")
print(f"Wrote {len(frags)} fragment SVGs into: {out_dir}")

Wrote: ibuprofen_brics_svgs/ibuprofen.svg
Wrote 3 fragment SVGs into: ibuprofen_brics_svgs


In [30]:
from rdkit.Chem import MolFromSmiles
other_fragments = [MolFromSmiles(s) for s in ['*CC', "*C(=O)C", "*C1=CC=CC=C1", "*C(C)C(=O)O"]]
for i, mol in enumerate(other_fragments):
    mol_to_svg_file(
        mol,
        out_dir / f"other_fragment_{i:02d}.svg",
        legend=f"Other frag {i}",
        size=(-1, -1),
    )
